# Student Summary Evaluation System
## AI-Powered Automated Assessment Framework

**Developer:** M Jagadeeswar Reddy  
**Roll Number:** 23BDS033  
**Project Date:** 15th November 2025

---

### Framework Architecture:
This intelligent assessment system performs the following operations:
1. Extracts lecture content from transcript documents
2. Analyzes student submissions using transformer-based NLP models
3. Computes multi-dimensional evaluation metrics (Content Coverage, Topic Relevance, Writing Clarity, Logical Coherence, Language Grammar)
4. Produces comprehensive scoring with detailed feedback (0-10 scale)
5. Automatically generates formatted Excel reports

## Phase 1: Dependencies Configuration
Setting up Python environment with required libraries

In [ ]:
# Package installation script
import subprocess
import sys

# Define required dependencies
required_packages = [
    'torch',
    'scikit-learn',
    'textstat',
    'nltk',
    'sentence-transformers',
    'python-docx',
    'openpyxl',
    'numpy',
    'pandas'
]

print("Initiating package installation...")
for pkg in required_packages:
    print(f"  → Installing {pkg}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print("✓ Package installation completed successfully!")

In [ ]:
# Library imports with reordered structure
import warnings
import os
from datetime import datetime
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import textstat
from nltk.tokenize import sent_tokenize
import nltk
from sentence_transformers import SentenceTransformer, util
from docx import Document
import numpy as np
import pandas as pd

# Suppress warning messages
warnings.filterwarnings('ignore')

# Initialize NLTK tokenizer resources
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

print("✓ Library imports successful!")
print(f"✓ Working directory: {os.getcwd()}")

## Phase 2: Neural Network Model Initialization
Configuring Sentence Transformer for semantic analysis

In [ ]:
# Initialize transformer model for semantic encoding
print("Initializing neural language model...")
semantic_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
print("✓ Model initialization complete!")
print(f"  Model architecture: paraphrase-MiniLM-L6-v2")
print(f"  Vector dimensions: {semantic_model.get_sentence_embedding_dimension()}")

## Phase 3: Data Ingestion
Extracting transcript and student submissions from source files

In [ ]:
def extract_transcript_content(docx_path):
    """
    Extract text content from Word document
    """
    try:
        document = Document(docx_path)
        content = "\n".join([para.text for para in document.paragraphs if para.text.strip()])
        print(f"✓ Transcript extracted: {len(content)} characters")
        return content
    except Exception as error:
        print(f"✗ Transcript extraction failed: {error}")
        return None

def extract_student_summaries(excel_path):
    """
    Extract student submissions from Excel spreadsheet
    """
    try:
        dataframe = pd.read_excel(excel_path)
        print(f"✓ Student data loaded: {len(dataframe)} entries")
        print(f"  Available columns: {list(dataframe.columns)}")
        return dataframe
    except Exception as error:
        print(f"✗ Summary extraction failed: {error}")
        return None

# Configure NLTK SSL for downloads
print("\n=== Configuring NLTK resources ===")
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Execute data loading
print("\n=== Data Extraction Process ===")
lecture_transcript = extract_transcript_content('transcript.docx')
student_data = extract_student_summaries('summary.xlsx')

if lecture_transcript and student_data is not None:
    print("\n✓ Data extraction completed successfully!")
    print(f"\nData preview:")
    print(student_data.head(3))

## Phase 4: Assessment Metrics Implementation
Building evaluation algorithms for multiple quality dimensions

In [ ]:
def identify_critical_sentences(text_content, sentence_count=10):
    """
    Extract most important sentences using TF-IDF scoring
    
    Args:
        text_content: Source text for analysis
        sentence_count: Number of sentences to extract
    
    Returns:
        list: Critical sentences identified
    """
    sentence_list = sent_tokenize(text_content)
    if len(sentence_list) <= sentence_count:
        return sentence_list
    
    # Apply TF-IDF vectorization
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_scores = tfidf_vectorizer.fit_transform(sentence_list)
    importance_scores = np.array(tfidf_scores.sum(axis=1)).flatten()
    critical_indices = importance_scores.argsort()[-sentence_count:][::-1]
    
    return [sentence_list[idx] for idx in sorted(critical_indices)]

def assess_content_coverage(student_summary, source_transcript, embedding_model):
    """
    Measure how comprehensively the summary captures transcript content
    Uses semantic similarity analysis with key points
    
    Returns:
        float: Coverage score (0-2 scale)
    """
    # Identify critical points from transcript
    critical_points = identify_critical_sentences(source_transcript, sentence_count=15)
    
    # Generate embeddings
    summary_vector = embedding_model.encode(student_summary, convert_to_tensor=True)
    critical_vectors = embedding_model.encode(critical_points, convert_to_tensor=True)
    
    # Compute similarity scores
    similarity_scores = util.cos_sim(summary_vector, critical_vectors)[0]
    
    # Calculate coverage metric
    high_match_count = (similarity_scores > 0.4).sum().item()
    coverage_percentage = high_match_count / len(critical_points)
    
    return min(2.0, coverage_percentage * 2.5)

def assess_topic_relevance(student_summary, source_transcript, embedding_model):
    """
    Evaluate alignment of summary with transcript topics
    Uses holistic semantic similarity
    
    Returns:
        float: Relevance score (0-2 scale)
    """
    # Generate document embeddings
    summary_vector = embedding_model.encode(student_summary, convert_to_tensor=True)
    transcript_vector = embedding_model.encode(source_transcript[:5000], convert_to_tensor=True)
    
    # Calculate semantic alignment
    alignment_score = util.cos_sim(summary_vector, transcript_vector)[0][0].item()
    
    # Convert to assessment scale
    return min(2.0, alignment_score * 2.5)

def assess_writing_clarity(student_summary):
    """
    Evaluate text readability and comprehension ease
    Uses Flesch Reading Ease metric
    
    Returns:
        float: Clarity score (0-2 scale)
    """
    try:
        # Calculate readability score
        readability_index = textstat.flesch_reading_ease(student_summary)
        
        # Map to assessment scale
        if readability_index >= 60:
            return 2.0
        elif readability_index >= 40:
            return 1.5
        elif readability_index >= 20:
            return 1.0
        else:
            return 0.5
    except:
        return 1.0

def assess_logical_coherence(student_summary):
    """
    Evaluate structural organization and logical flow
    Analyzes sentence distribution and transition usage
    
    Returns:
        float: Coherence score (0-2 scale)
    """
    sentence_list = sent_tokenize(student_summary)
    coherence_score = 0.0
    
    # Evaluate sentence count balance
    if 3 <= len(sentence_list) <= 8:
        coherence_score += 1.0
    elif len(sentence_list) >= 2:
        coherence_score += 0.5
    
    # Detect transition phrases
    transition_indicators = ['however', 'therefore', 'additionally', 'furthermore', 'moreover',
                           'consequently', 'thus', 'also', 'first', 'second', 'finally', 'next']
    lowercase_summary = student_summary.lower()
    transition_count = sum(1 for indicator in transition_indicators if indicator in lowercase_summary)
    
    if transition_count >= 2:
        coherence_score += 1.0
    elif transition_count >= 1:
        coherence_score += 0.5
    
    return min(2.0, coherence_score)

def assess_language_grammar(student_summary):
    """
    Evaluate grammatical correctness and language quality
    Performs pattern-based quality checks
    
    Returns:
        float: Grammar score (0-2 scale)
    """
    quality_score = 2.0
    
    # Check for excessive spacing
    if re.search(r'\s{2,}', student_summary):
        quality_score -= 0.2
    
    # Verify sentence capitalization
    sentence_list = sent_tokenize(student_summary)
    improper_caps = sum(1 for sent in sentence_list if sent and not sent[0].isupper())
    quality_score -= improper_caps * 0.2
    
    # Check sentence termination
    missing_punctuation = sum(1 for sent in sentence_list if sent and sent[-1] not in '.!?')
    quality_score -= missing_punctuation * 0.2
    
    # Evaluate word quality metrics
    try:
        word_length_avg = textstat.avg_character_per_word(student_summary)
        if word_length_avg < 3:
            quality_score -= 0.3
    except:
        pass
    
    return max(0.5, min(2.0, quality_score))

print("✓ Assessment functions configured successfully!")

## Phase 5: Integrated Evaluation Engine
Comprehensive scoring system with feedback generation

In [ ]:
def perform_comprehensive_evaluation(student_summary, source_transcript, embedding_model):
    """
    Execute complete multi-dimensional assessment of student work
    
    Args:
        student_summary: Text submitted by student
        source_transcript: Original lecture content
        embedding_model: Neural network model for semantic analysis
    
    Returns:
        dict: Comprehensive evaluation results with scores and feedback
    """
    # Validate input
    if not student_summary or not isinstance(student_summary, str) or len(student_summary.strip()) < 20:
        return {
            'score': 0.0,
            'coverage': 0.0,
            'relevance': 0.0,
            'clarity': 0.0,
            'coherence': 0.0,
            'grammar': 0.0,
            'explanation': 'Invalid or missing summary provided.'
        }
    
    # Execute dimensional assessments
    coverage_rating = assess_content_coverage(student_summary, source_transcript, embedding_model)
    relevance_rating = assess_topic_relevance(student_summary, source_transcript, embedding_model)
    clarity_rating = assess_writing_clarity(student_summary)
    coherence_rating = assess_logical_coherence(student_summary)
    grammar_rating = assess_language_grammar(student_summary)
    
    # Aggregate total score
    aggregate_score = coverage_rating + relevance_rating + clarity_rating + coherence_rating + grammar_rating
    
    # Generate detailed feedback
    feedback_components = []
    
    # Coverage analysis
    if coverage_rating >= 1.5:
        feedback_components.append("Demonstrates excellent mastery of key lecture concepts")
    elif coverage_rating >= 1.0:
        feedback_components.append("Captures main ideas adequately but omits certain important elements")
    else:
        feedback_components.append("Limited representation of core lecture material")
    
    # Relevance analysis
    if relevance_rating >= 1.5:
        feedback_components.append("shows strong alignment with transcript themes")
    elif relevance_rating >= 1.0:
        feedback_components.append("generally relevant with occasional off-topic elements")
    else:
        feedback_components.append("deviates from primary lecture focus")
    
    # Writing quality analysis
    if clarity_rating >= 1.5 and grammar_rating >= 1.5:
        feedback_components.append("Exhibits strong writing with excellent clarity and grammatical accuracy")
    elif clarity_rating >= 1.0 or grammar_rating >= 1.0:
        feedback_components.append("Shows acceptable writing quality with some areas for improvement")
    else:
        feedback_components.append("Requires substantial improvement in writing mechanics and clarity")
    
    # Coherence analysis
    if coherence_rating < 1.0:
        feedback_components.append("Would benefit from better organization and logical progression")
    
    comprehensive_feedback = ". ".join(feedback_components) + "."
    
    return {
        'score': round(aggregate_score, 2),
        'coverage': round(coverage_rating, 2),
        'relevance': round(relevance_rating, 2),
        'clarity': round(clarity_rating, 2),
        'coherence': round(coherence_rating, 2),
        'grammar': round(grammar_rating, 2),
        'explanation': comprehensive_feedback
    }

print("✓ Integrated evaluation engine ready!")

## Phase 6: Batch Processing Pipeline
Processing all student submissions with progress monitoring

In [ ]:
def process_all_student_submissions(student_dataframe, lecture_content, neural_model):
    """
    Batch process and evaluate all student summaries.
    
    Args:
        student_dataframe: DataFrame containing student submissions
        lecture_content: Reference transcript text
        neural_model: Embedding model for semantic analysis
    
    Returns:
        DataFrame: Complete evaluation results with scores and feedback
    """
    evaluation_results = []
    total_submissions = len(student_dataframe)
    
    print("\nStarting Batch Evaluation")
    print(f"Total submissions to evaluate: {total_submissions}\n")
    
    # Identify the summary column
    summary_column_name = None
    for column in student_dataframe.columns:
        if 'summary' in column.lower():
            summary_column_name = column
            break
    
    if summary_column_name is None:
        print("Error: No summary column found in the dataset.")
        return None
    
    # Process each student submission
    for index, student_row in student_dataframe.iterrows():
        current_position = index + 1
        student_name = student_row.get('Name of the Student', 'Unknown')
        
        print(f"Processing submission {current_position} of {total_submissions}")
        print(f"Student: {student_name}")
        
        submission_text = student_row[summary_column_name]
        
        # Perform evaluation
        assessment_result = perform_comprehensive_evaluation(
            submission_text,
            lecture_content,
            neural_model
        )
        
        # Construct result entry
        result_entry = {
            'Email Address': student_row.get('Email Address', ''),
            'Name': student_row.get('Name of the Student', ''),
            'Roll Number': student_row.get('Roll number', ''),
            'Institute': student_row.get('Institute', ''),
            'Summary': submission_text,
            'Score': assessment_result['score'],
            'Coverage': assessment_result['coverage'],
            'Relevance': assessment_result['relevance'],
            'Clarity': assessment_result['clarity'],
            'Coherence': assessment_result['coherence'],
            'Grammar': assessment_result['grammar'],
            'Explanation': assessment_result['explanation']
        }
        
        evaluation_results.append(result_entry)
        
        print(f"Final Score: {assessment_result['score']}/10\n")
    
    print("Batch Evaluation Completed\n")
    print(f"Total evaluations processed: {total_submissions}\n")
    
    return pd.DataFrame(evaluation_results)

# Execute batch evaluation
if lecture_transcript and student_data is not None:
    evaluation_dataframe = process_all_student_submissions(
        student_data, 
        lecture_transcript, 
        semantic_model
    )
else:
    print("Cannot proceed: Required data not loaded.")


## Phase 7: Results Visualization
Statistical analysis and performance rankings

In [ ]:
if evaluation_dataframe is not None:
    print("\nSTATISTICAL SUMMARY")
    print("-------------------------------------------------------------\n")
    
    print(f"Total Submissions Assessed: {len(evaluation_dataframe)}")
    print(f"Mean Score: {evaluation_dataframe['Score'].mean():.2f}/10")
    print(f"Maximum Score: {evaluation_dataframe['Score'].max():.2f}/10")
    print(f"Minimum Score: {evaluation_dataframe['Score'].min():.2f}/10")
    print(f"Score Standard Deviation: {evaluation_dataframe['Score'].std():.2f}")
    
    print("\nDimensional Performance Averages")
    print("-------------------------------------------------------------")
    print(f"  Content Coverage (0-2):    {evaluation_dataframe['Coverage'].mean():.2f}")
    print(f"  Topic Relevance (0-2):     {evaluation_dataframe['Relevance'].mean():.2f}")
    print(f"  Writing Clarity (0-2):     {evaluation_dataframe['Clarity'].mean():.2f}")
    print(f"  Logical Coherence (0-2):   {evaluation_dataframe['Coherence'].mean():.2f}")
    print(f"  Language Grammar (0-2):    {evaluation_dataframe['Grammar'].mean():.2f}")
    
    print("\nTOP 5 PERFORMERS")
    print("-------------------------------------------------------------\n")
    
    top_performers = evaluation_dataframe.nlargest(5, 'Score')[['Name', 'Roll Number', 'Score', 'Explanation']]
    
    for idx, performance_row in top_performers.iterrows():
        rank_position = list(top_performers.index).index(idx) + 1
        print(f"Position {rank_position}:")
        print(f"  Student: {performance_row['Name']}")
        print(f"  Roll No: {performance_row['Roll Number']}")
        print(f"  Score: {performance_row['Score']}/10")
        print(f"  Assessment: {performance_row['Explanation']}")
        print("-------------------------------------------------------------")
    
    print("\nResults visualization is complete.\n")


## Phase 8: Excel Report Generation
Exporting formatted results to spreadsheet

In [ ]:
def generate_excel_report(results_dataframe, student_roll='23BDS033'):
    """
    Generate formatted Excel report with evaluation results
    
    Args:
        results_dataframe: DataFrame with complete evaluation data
        student_roll: Roll number for report filename
    """
    report_filename = f'Graded_results_{student_roll}.xlsx'
    
    try:
        # Sort results by performance
        sorted_results = results_dataframe.sort_values('Score', ascending=False).reset_index(drop=True)
        
        # Initialize Excel writer
        with pd.ExcelWriter(report_filename, engine='openpyxl') as excel_writer:
            # Export main results
            sorted_results.to_excel(excel_writer, sheet_name='Graded Results', index=False)
            
            # Access workbook components
            workbook_obj = excel_writer.book
            worksheet_obj = excel_writer.sheets['Graded Results']
            
            # Configure column dimensions
            column_width_config = {
                'A': 30,
                'B': 25,
                'C': 15,
                'D': 20,
                'E': 50,
                'F': 10,
                'G': 12,
                'H': 12,
                'I': 12,
                'J': 12,
                'K': 12,
                'L': 60
            }
            
            for column_id, width_value in column_width_config.items():
                worksheet_obj.column_dimensions[column_id].width = width_value
            
            # Create statistical summary sheet
            statistics_data = {
                'Metric': [
                    'Total Students', 'Average Score', 'Highest Score', 'Lowest Score',
                    'Standard Deviation', 'Avg Coverage', 'Avg Relevance', 
                    'Avg Clarity', 'Avg Coherence', 'Avg Grammar'
                ],
                'Value': [
                    len(sorted_results),
                    f"{sorted_results['Score'].mean():.2f}/10",
                    f"{sorted_results['Score'].max():.2f}/10",
                    f"{sorted_results['Score'].min():.2f}/10",
                    f"{sorted_results['Score'].std():.2f}",
                    f"{sorted_results['Coverage'].mean():.2f}/2",
                    f"{sorted_results['Relevance'].mean():.2f}/2",
                    f"{sorted_results['Clarity'].mean():.2f}/2",
                    f"{sorted_results['Coherence'].mean():.2f}/2",
                    f"{sorted_results['Grammar'].mean():.2f}/2"
                ]
            }
            
            stats_dataframe = pd.DataFrame(statistics_data)
            stats_dataframe.to_excel(excel_writer, sheet_name='Statistics', index=False)
        
        # Clean, professional output
        print("\nExcel Report Export")
        print("-------------------------------------------------------------")
        print(f"Report generated successfully: {report_filename}")
        print(f"Location: {os.path.abspath(report_filename)}")
        print(f"Total records exported: {len(sorted_results)}")
        print("Sheets included: Graded Results, Statistics")
        print("-------------------------------------------------------------\n")
        
        return report_filename
    
    except Exception as error:
        print("\nExport failed.")
        print("-------------------------------------------------------------")
        print(f"Reason: {error}")
        print("-------------------------------------------------------------\n")
        return None

# Generate Excel output
if evaluation_dataframe is not None:
    report_output = generate_excel_report(evaluation_dataframe, student_roll='23BDS033')
else:
    print("No data available for export.")


## Phase 9: System Summary
Execution report and deployment verification

In [ ]:
print("\nSTUDENT SUMMARY EVALUATION SYSTEM - COMPLETE")
print("-------------------------------------------------------------\n")

print("System executed all phases successfully:\n")
print("  1. Extracted lecture content from transcript.docx")
print("  2. Imported student submissions from summary.xlsx")
print("  3. Initialized neural network model (Sentence Transformer)")
print("  4. Performed multi-dimensional evaluation:")
print("       • Content Coverage - Key point identification")
print("       • Topic Relevance - Semantic alignment")
print("       • Writing Clarity - Readability assessment")
print("       • Logical Coherence - Structural analysis")
print("       • Language Grammar - Quality verification")
print("  5. Computed comprehensive scores (0-10 scale) with detailed feedback")
print("  6. Generated Excel report: Graded_results_23BDS033.xlsx")
print("  7. Presented top performer rankings for validation\n")

if evaluation_dataframe is not None:
    print("Assessment Statistics:")
    print(f"   • Total submissions evaluated: {len(evaluation_dataframe)}")
    print(f"   • Mean performance score: {evaluation_dataframe['Score'].mean():.2f}/10")
    print(f"   • Performance range: {evaluation_dataframe['Score'].min():.2f} - {evaluation_dataframe['Score'].max():.2f}")

print("\nSYSTEM DEPLOYMENT VERIFICATION PASSED")
print("-------------------------------------------------------------\n")

print("Demonstration Readiness:")
print("   • Input data successfully loaded and validated")
print("   • AI evaluation pipeline operational")
print("   • Scoring and feedback generation verified")
print("   • Excel output generated in working directory")
print("   • Top performer rankings displayed")
print("\n   System ready for demonstration!\n")

print("-------------------------------------------------------------")
print("To rerun: Execute 'Run All' or restart kernel and run all cells sequentially")
print("-------------------------------------------------------------")
